# Patnaik-Pearson meets BERT internal representations

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy.linalg import qr

import math
import pandas as pd
import random

import torch

In [ ]:
!pip install cupy-cuda12x

In [2]:
import PatnaikPearson as pp
import cupy

In [ ]:
!nvidia-smi

In [ ]:
# !pip install transformers
#!pip install transformers accelerate
#!pip install transformers==4.51.3
!pip install transformers==4.44.0 accelerate==0.33.0

In [19]:
# new - disable SDPA (Scaled Dot Product Attention)

from transformers import BertModel, BertConfig, BertTokenizer

# Load model with SDPA disabled
model = BertModel.from_pretrained(
    'bert-base-uncased',
    output_hidden_states=True,
    attn_implementation='eager'   # disables SDPA, uses classic attention
)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model.eval()

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [4]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

4.44.0
C:\Users\Dell\OneDrive\Documents\TomDocuments\PythonProjects\PP\20260527\PatnaikPearson\ml_env311_v2\Lib\site-packages\transformers\__init__.py


In [ ]:
import torch
print(torch.__version__)

In [ ]:
from transformers import BertModel, BertConfig
print("import OK")

# PP dim of token embeddings

In [5]:
valid_embeddings, invalid_embeddings = pp.get_valid_invalid_bert_base_token_embeddings(model)

In [ ]:
print(valid_embeddings.shape)
print(invalid_embeddings.shape)

In [6]:
pp_dim_inv_emb = pp.calculate_PatnaikPearson_dim(invalid_embeddings, verbose=False)
print("pp_dim_inv_emb = ", pp_dim_inv_emb)

pp_dim_inv_emb =  561.742428250816


In [ ]:
avg = np.mean(invalid_embeddings, axis=0)
print(avg.shape)

print(math.sqrt(np.dot(avg,avg)))
#print(math.norm(avg))

demeaned = invalid_embeddings - avg
print(demeaned.shape)

In [7]:
num_embeddings = 2000
num_valid_embeddings = valid_embeddings.shape[0] #len(valid_ids)
print("num_valid_embeddings =", num_valid_embeddings)
num_embeddings = min(num_embeddings, num_valid_embeddings)
print("num_embeddings =", num_embeddings)
embedding_dim = valid_embeddings.shape[1]
print("embedding_dim =", embedding_dim)

random_indices = np.random.choice(num_valid_embeddings, num_embeddings, replace = False)
print(random_indices.shape)

X = np.zeros((num_embeddings,768))
for i in range(num_embeddings):
  this_index = random_indices[i]
  X[i,:] = valid_embeddings[this_index]

print("X.shape = ", X.shape)

pp_dim_X = pp.calculate_PatnaikPearson_dim(X, verbose=False)
dim_X = X.shape[1]
nu_over_d_X = pp_dim_X / dim_X
print("pp_dim_X = ", pp_dim_X)
print("nu_over_d_X = ", nu_over_d_X)


num_valid_embeddings = 29528
num_embeddings = 2000
embedding_dim = 768
(2000,)
X.shape =  (2000, 768)
pp_dim_X =  600.9224614488321
nu_over_d_X =  0.7824511216781668


In [8]:
pp.analyse_embeddings(valid_embeddings)

these_embeddings.shape =  (29528, 768)
num_embeddings =  29528
raw embedding norm stats:
shape : (29528,)
min : 0.7657115047329871
max : 2.0445971930627644
sum : 41620.029288117184
mean : 1.4095106098657946
std : 0.191249034702401
norm : 244.42564247545445
norm(avg_embedding) : 0.9373921515531835
sanity check - should be zero :  2.0140286166514216e-27
raw demeaned_embedding norm stats:
shape : (29528,)
min : 0.5759314096862294
max : 2.1275616389432797
sum : 31302.248908141053
mean : 1.0600869990565245
std : 0.14424930237845343
norm : 183.84102272166575
num_iterations =  14
0 pp_dim_initial =  600.3859475766902
0 pp_dim_demeaned =  600.3859475766899
1 pp_dim_initial =  603.4586177397009
1 pp_dim_demeaned =  603.458617739701
2 pp_dim_initial =  602.9091566769871
2 pp_dim_demeaned =  602.9091566769875
3 pp_dim_initial =  603.2694426648897
3 pp_dim_demeaned =  603.2694426648894
4 pp_dim_initial =  604.9732531962665
4 pp_dim_demeaned =  604.9732531962667
5 pp_dim_initial =  602.212142203262

In [10]:
valid_bert_base_token_ids = pp.get_valid_bert_base_token_ids()
print(len(valid_bert_base_token_ids))

29528


In [21]:
num_samples = 1000
sampled_token_ids = np.random.choice(valid_bert_base_token_ids, size=num_samples, replace=False)
print(type(sampled_token_ids))
print(sampled_token_ids.shape)

sampled_token_ids_list = sampled_token_ids.tolist()
print(type(sampled_token_ids_list))

X0 = tokenizer.convert_ids_to_tokens(sampled_token_ids_list) 
print(type(X0))
print(X0[0:5])

<class 'numpy.ndarray'>
(1000,)
<class 'list'>
<class 'list'>
['thessaloniki', '##hr', 'calcutta', 'gujarati', 'agitated']


In [23]:
N = len(X0)
print("N = ", N)

# Build input: shape (1, N) — single "sentence" of N tokens
input_ids      = (torch.tensor(sampled_token_ids)).unsqueeze(0).to(DEVICE)          # (1, N)
attention_mask = torch.ones(1, N, dtype=torch.long).to(DEVICE)     # no padding


N =  1000


NameError: name 'DEVICE' is not defined